In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

In [5]:
d = pd.read_csv('global_space_launch_missions.csv')
d.head()

,Mission_ID,Mission_Name,Launch_Date,Launch_Year,Country,Launch_Site,Organization,Rocket_Name,Rocket_Family,Rocket_Height_m,...,Mission_Type,Launch_Azimuth_Degrees,Temperature_C,Humidity_Percent,Wind_Speed_mps,Launch_Delay_Days,Rocket_Reused,Previous_Rocket_Success_Rate,Mission_Duration_Hours,Launch_Cost_Million_USD
0,MIS-10000,Hayabusa-IV,2018-03-18,2018,USA,Vandenberg SFB SLC-4E,NASA,SLS Block 1,SLS,99.46,...,Cargo,85.56,20.5,28.7,1.4,NaN,No,83.26,33.6,282.72
1,MIS-10001,Ariane-Satellite,2019-11-02,2019,USA,Cape Canaveral SLC-40,NASA,SLS Block 1,SLS,102.58,...,Deep Space,54.68,3.4,NaN,0.4,NaN,No,86.20,67.4,247.85
2,MIS-10002,Galileo-Beta,2012-10-01,2012,USA,Kennedy Space Center LC-39A,NASA,SLS Block 1,SLS,99.06,...,Satellite Deployment,48.53,33.0,53.3,0.9,0.0,No,93.37,139.8,309.82
3,MIS-10003,Ariane-V,2025-07-08,2025,China,Wenchang Space Launch Site,CNSA,Long March 2F,Long March,47.23,...,Cargo,90.52,19.3,61.5,9.5,1.0,No,93.51,4.0,30.62
4,MIS-10004,Hayabusa-Prime,2018-07-26,2018,Japan,Tanegashima Space Center,JAXA,H3,H3,62.46,...,Navigation,65.64,NaN,98.4,4.6,NaN,No,85.91,44.9,60.02


In [6]:
# 1. Define features and target (dropping leaky or unique identifiers)
X = d.drop(columns=['Mission_ID', 'Mission_Name', 'Launch_Date', 'Launch_Cost_Million_USD'])
y = d['Launch_Cost_Million_USD']

In [7]:
# 2. Split data FIRST to avoid data leakage
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
# 3. Identify numeric and categorical columns dynamically
numeric_cols = x_train.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = x_train.select_dtypes(include=['object', 'category']).columns

In [9]:
# 4. Create preprocessing pipelines for both data types
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

In [10]:
# 5. Combine preprocessor and model into a full pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])

In [11]:
# 6. Train and evaluate
pipeline.fit(x_train, y_train)
y_pred = pipeline.predict(x_test)

print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

RMSE: 62.19997292625038
R2 Score: 0.8213676104614804


In [12]:
joblib.dump(pipeline, 'space_launch_cost_model.pkl')

['space_launch_cost_model.pkl']

In [13]:
import pandas as pd

# Make sure all training columns are present here!
new_mission_data = pd.DataFrame({
    'Launch_Year': [2026],
    'Country': ['USA'],
    'Launch_Site': ['Kennedy Space Center LC-39A'],
    'Organization': ['SpaceX'],
    'Rocket_Name': ['Falcon 9'],
    'Rocket_Family': ['Falcon'],
    'Rocket_Height_m': [70.0],
    'Rocket_Mass_tons': [549.0],
    'Payload_Mass_kg': [15500.0],
    'Number_of_Stages': [2],
    'Fuel_Type': ['Liquid Oxygen/RP-1'],
    'Orbit_Type': ['LEO'],
    'Mission_Type': ['Satellite Deployment'],
    'Launch_Azimuth_Degrees': [45.0],
    'Temperature_C': [25.0],
    'Humidity_Percent': [60.0],
    'Wind_Speed_mps': [3.5],
    'Launch_Delay_Days': [0.0],
    'Rocket_Reused': ['Yes'],
    'Previous_Rocket_Success_Rate': [98.5],
    'Mission_Duration_Hours': [24.0],
    'Booster_Count': [1]  # <--- Added this missing column!
})

# Now run the prediction again
predicted_cost = pipeline.predict(new_mission_data)
print(f"Estimated Launch Cost: ${predicted_cost[0]:.2f} Million USD")

Estimated Launch Cost: $172.78 Million USD


C:\Users\Pavan\AppData\Roaming\Python\Python313\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [5] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
